# Frobenius Motif Similarity Analysis

Visually cluster the cropped motif images in `frobenius_artifacts/analysis/motifs/` by appearance.

**Pipeline:**
1. Scan motif crops and load metadata from filenames
2. Embed every crop with **CLIP ViT-B/32** (512-dim, cosine space)
3. Cache embeddings so subsequent runs are instant
4. Project to 2-D with **t-SNE** for layout
5. Cluster with **HDBSCAN** for groupings
6. **Scatter map** — every motif as a thumbnail, neighbours are visually similar
7. **Cluster gallery** — grids of same-cluster motifs
8. **Cosine similarity heatmap**
9. **Nearest-neighbour explorer** — pick any motif, see its closest matches

```bash
uv run --project src/python jupyter notebook src/python/motif_similarity.ipynb
```

> Run `extract_crops.py` first if the motifs directory is empty.

In [1]:
# ── Cell 1: Colab setup (skip locally) ────────────────────────────────────
import sys
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "open-clip-torch", "hdbscan", "scikit-learn",
        "Pillow", "numpy>=1.24,<2", "ipywidgets",
    ], check=True)
    print("Colab: deps installed.")
else:
    print("Local — skipping Colab setup.")

Local — skipping Colab setup.


In [2]:
# ── Cell 2: imports & paths ────────────────────────────────────────────────
import sys, os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

import torch
import open_clip

from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import hdbscan

# ── Paths ──────────────────────────────────────────────────────────────────
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    MOTIFS_DIR = Path("/content/motifs")
    CACHE_DIR  = Path("/content")
else:
    REPO_ROOT  = Path("../..")
    MOTIFS_DIR = REPO_ROOT / "frobenius_artifacts/analysis/motifs"
    CACHE_DIR  = Path(".")   # src/python/ — keeps cache next to notebook

EMBED_CACHE = CACHE_DIR / "motif_embeddings.npy"
PATHS_CACHE = CACHE_DIR / "motif_paths.txt"

print(f"Motifs dir : {MOTIFS_DIR.resolve()}")
print(f"Cache dir  : {CACHE_DIR.resolve()}")
if not MOTIFS_DIR.exists():
    print("WARNING: motifs directory not found — run extract_crops.py first.")

Motifs dir : /Users/korede/code/surulere/african-artifacts/frobenius_artifacts/analysis/motifs
Cache dir  : /Users/korede/code/surulere/african-artifacts/src/python


In [3]:
# ── Cell 3: scan motif crops & build metadata ──────────────────────────────
# Expected layout:
#   motifs/<panel_stem>/<NNN>_<scale>_iou<X.XXX>.png

records = []
for panel_dir in sorted(MOTIFS_DIR.iterdir()):
    if not panel_dir.is_dir():
        continue
    panel_name = panel_dir.name
    for img_path in sorted(panel_dir.glob("*.png")):
        stem   = img_path.stem          # e.g. "003_motif_iou0.889"
        parts  = stem.split("_")
        idx    = int(parts[0])
        scale  = parts[1]               # motif | register
        iou    = float(parts[2].replace("iou", ""))
        records.append({
            "path"    : img_path,
            "panel"   : panel_name,
            "index"   : idx,
            "scale"   : scale,
            "pred_iou": iou,
        })

panels = sorted(set(r["panel"] for r in records))
print(f"{len(records)} crops across {len(panels)} panels")
for p in panels:
    n = sum(1 for r in records if r["panel"] == p)
    print(f"  {p}: {n}")

268 crops across 9 panels
  EBA-B_00425_Ibadan_q97912_i1_panel_0_cropped: 44
  EBA-Div_00303_Ado_Ekiti_q166559_i1_panel_00: 53
  EBA-Div_00311_Ife_q166566_i1_panel_00: 29
  EBA-Div_00311_Ife_q166566_i1_panel_01: 20
  EBA-Div_00311_Ife_q166566_i1_panel_02: 24
  EBA-Div_00312_Ife_q166567_i1_panel_00: 26
  EBA-Div_00312_Ife_q166567_i1_panel_01: 25
  FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_00: 35
  FoA_04-5578_Modakeke_(Ife)_q48628_i1_panel_01: 12


In [ ]:
# ── Cell 4: CLIP embeddings (cached, with preprocessing) ──────────────────
#
# PREPROCESS_MODE controls what is fed into CLIP:
#   "color"     — raw crop (CLIP default; biased toward tint/hue)
#   "grayscale" — strips all color; retains texture but not tone
#   "edges"     — Canny edge map; focuses purely on shape contours
#   "clahe"     — per-channel contrast normalisation; reduces brightness bias
#
# Change the mode and re-run this cell to recompute embeddings.
# Each mode caches separately so switching is fast after the first run.

import cv2 as _cv2

PREPROCESS_MODE = "grayscale"   # ← change me: color | grayscale | edges | clahe
CLIP_MODEL      = "ViT-B-32"
CLIP_WEIGHTS    = "openai"
BATCH           = 32

EMBED_CACHE = CACHE_DIR / f"motif_embeddings_{PREPROCESS_MODE}.npy"
PATHS_CACHE = CACHE_DIR / f"motif_paths_{PREPROCESS_MODE}.txt"


def preprocess_image(img: "PIL.Image") -> "PIL.Image":
    """Apply PREPROCESS_MODE transform before CLIP preprocessing."""
    img = img.convert("RGB")
    arr = np.array(img)

    if PREPROCESS_MODE == "color":
        return img

    elif PREPROCESS_MODE == "grayscale":
        gray = _cv2.cvtColor(arr, _cv2.COLOR_RGB2GRAY)
        return Image.fromarray(np.stack([gray] * 3, axis=-1))

    elif PREPROCESS_MODE == "edges":
        gray   = _cv2.cvtColor(arr, _cv2.COLOR_RGB2GRAY)
        # Adaptive Canny: auto-threshold from median intensity
        med    = float(np.median(gray))
        lo, hi = max(0, 0.5 * med), min(255, 1.5 * med)
        edges  = _cv2.Canny(gray, lo, hi)
        # White edges on black — invert so lines are dark on white (CLIP-friendlier)
        edges  = 255 - edges
        return Image.fromarray(np.stack([edges] * 3, axis=-1))

    elif PREPROCESS_MODE == "clahe":
        lab  = _cv2.cvtColor(arr, _cv2.COLOR_RGB2LAB).astype(np.uint8)
        cl   = _cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
        lab[:, :, 0] = cl.apply(lab[:, :, 0])
        return Image.fromarray(_cv2.cvtColor(lab, _cv2.COLOR_LAB2RGB))

    else:
        raise ValueError(f"Unknown PREPROCESS_MODE: {PREPROCESS_MODE!r}")


def _resolve_device():
    if torch.cuda.is_available(): return "cuda"
    return "cpu"

_clip_cache = {}

def get_clip():
    if not _clip_cache:
        device = _resolve_device()
        print(f"Loading CLIP {CLIP_MODEL} ({CLIP_WEIGHTS}) on {device}...")
        model, _, prep = open_clip.create_model_and_transforms(
            CLIP_MODEL, pretrained=CLIP_WEIGHTS
        )
        model = model.to(device).eval()
        _clip_cache.update({"model": model, "prep": prep, "device": device})
        print("CLIP ready.")
    return _clip_cache["model"], _clip_cache["prep"], _clip_cache["device"]


def compute_embeddings(recs):
    model, prep, device = get_clip()
    all_feats = []
    for i in range(0, len(recs), BATCH):
        batch = recs[i : i + BATCH]
        imgs  = torch.stack([
            prep(preprocess_image(Image.open(r["path"])))
            for r in batch
        ]).to(device)
        with torch.no_grad():
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        all_feats.append(feats.cpu().float().numpy())
        print(f"  {min(i + BATCH, len(recs))}/{len(recs)}", end="\r")
    print()
    return np.vstack(all_feats)


current_paths = [str(r["path"]) for r in records]

if (
    EMBED_CACHE.exists() and PATHS_CACHE.exists()
    and PATHS_CACHE.read_text().strip().split("\n") == current_paths
):
    print(f"Loading cached embeddings ({PREPROCESS_MODE})...")
    embeddings = np.load(EMBED_CACHE)
    print(f"  shape: {embeddings.shape}")
else:
    print(f"Computing CLIP embeddings (mode={PREPROCESS_MODE})...")
    embeddings = compute_embeddings(records)
    np.save(EMBED_CACHE, embeddings)
    PATHS_CACHE.write_text("\n".join(current_paths))
    print(f"Computed and cached: {embeddings.shape}")

print(f"\nPreprocess mode : {PREPROCESS_MODE}")
print(f"Embedding shape : {embeddings.shape}")


In [ ]:
# ── Cell 5: t-SNE projection + interactive HDBSCAN re-clustering ───────────
#
# t-SNE runs once (slow); clustering is instant and re-runs live via sliders.
#
# HDBSCAN knobs:
#   min_cluster_size  — minimum members for a group to count as a cluster.
#                       Lower = more, smaller clusters. Start at 3–5.
#   min_samples       — how conservative the core-point definition is.
#                       1 = most permissive (fewer noise points).
#   method            — "leaf" finds finer splits than "eom" (excess of mass).

N = len(records)

# ── t-SNE (run once) ──────────────────────────────────────────────────────
PERPLEXITY = min(30, max(5, N // 8))
print(f"Running t-SNE (n={N}, perplexity={PERPLEXITY})...")

import sklearn as _sklearn
_sk_ver  = tuple(int(x) for x in _sklearn.__version__.split(".")[:2])
_iter_kw = "max_iter" if _sk_ver >= (1, 5) else "n_iter"

tsne   = TSNE(n_components=2, perplexity=PERPLEXITY, metric="euclidean",
              random_state=42, init="pca", **{_iter_kw: 1500})
coords = tsne.fit_transform(embeddings)
print(f"t-SNE done.")

# ── Cosine similarity matrix ──────────────────────────────────────────────
sim_matrix = cosine_similarity(embeddings)

# ── Interactive re-clustering widget ─────────────────────────────────────
w_mcs = widgets.IntSlider(
    min=2, max=20, step=1, value=3,
    description="min_cluster_size",
    continuous_update=False,
    style={"description_width": "150px"},
    layout=widgets.Layout(width="60%"),
)
w_ms = widgets.IntSlider(
    min=1, max=10, step=1, value=1,
    description="min_samples",
    continuous_update=False,
    style={"description_width": "150px"},
    layout=widgets.Layout(width="60%"),
)
w_method = widgets.ToggleButtons(
    options=["leaf", "eom"],
    value="leaf",
    description="selection method",
    style={"description_width": "150px", "button_width": "80px"},
)
out_cluster_summary = widgets.Output()

# Shared mutable state — updated by the widget callback
_state = {"labels": None, "n_clusters": 0, "n_noise": 0}

def run_clustering(mcs, ms, method):
    cl = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric="euclidean",
        cluster_selection_method=method,
    )
    lbl = cl.fit_predict(embeddings)
    nc  = len(set(lbl)) - (1 if -1 in lbl else 0)
    nn  = int((lbl == -1).sum())
    _state["labels"]    = lbl
    _state["n_clusters"] = nc
    _state["n_noise"]    = nn

    out_cluster_summary.clear_output(wait=True)
    with out_cluster_summary:
        print(f"→ {nc} clusters, {nn}/{N} unclustered (noise)  "
              f"[mode={PREPROCESS_MODE}, mcs={mcs}, ms={ms}, method={method}]")
        for c in sorted(set(lbl)):
            tag = "  noise" if c == -1 else f"  C{c:2d}"
            bar = "█" * min(40, int((lbl == c).sum()))
            print(f"{tag}: {(lbl == c).sum():3d}  {bar}")

def _on_cluster_change(_=None):
    run_clustering(w_mcs.value, w_ms.value, w_method.value)

w_mcs.observe(_on_cluster_change, names="value")
w_ms.observe(_on_cluster_change, names="value")
w_method.observe(_on_cluster_change, names="value")

display(
    widgets.HTML("<b>HDBSCAN parameters</b> — adjust and the cluster summary updates instantly:"),
    w_mcs, w_ms, w_method,
    out_cluster_summary,
)

# Run with defaults
run_clustering(w_mcs.value, w_ms.value, w_method.value)

# Convenience accessor used by cells 6–9
def labels():
    return _state["labels"]


In [ ]:
# ── Cell 6: similarity map — thumbnails on t-SNE axes ─────────────────────
# Re-run this cell after adjusting clustering in cell 5.
#
# Each motif is drawn at its t-SNE position as a thumbnail with:
#   - Coloured border = cluster
#   - Index label     = motif number within its panel
#   - "P" abbreviation from the panel name shown below each image
#
# The figure is saved to motif_scatter_<mode>.png alongside the notebook.

lbl = labels()   # pick up current clustering from cell 5

THUMB  = 64      # px — larger = more detail, more overlap in dense regions
ZOOM   = 0.72    # OffsetImage zoom (THUMB * ZOOM ≈ rendered size in data units)
FIG_W  = 26
FIG_H  = 20
DPI    = 130

_PALETTE   = list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors)
_NOISE_COL = (0.45, 0.45, 0.45)

def cluster_color(l):
    return _NOISE_COL if l == -1 else _PALETTE[l % len(_PALETTE)]

# Short panel label: strip long common prefix, keep meaningful tail
_panel_names = sorted(set(r["panel"] for r in records))

def _short_panel(name):
    # Try to extract the city/site token from the filename
    # e.g. "EBA-B_00425_Ibadan_q97912_i1_panel_0_cropped" → "Ibadan"
    for tok in name.split("_"):
        if tok and tok[0].isupper() and tok not in ("EBA", "Div", "FoA", "Ife"):
            return tok[:8]
    return name[-8:]

# Load + square-pad thumbnails
print("Loading thumbnails...")
thumbs = []
for r in records:
    img = Image.open(r["path"]).convert("RGB")
    img.thumbnail((THUMB, THUMB))
    sq = Image.new("RGB", (THUMB, THUMB), (25, 25, 25))
    sq.paste(img, ((THUMB - img.width) // 2, (THUMB - img.height) // 2))
    thumbs.append(np.array(sq))

# ── Draw scatter ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), dpi=DPI)
ax.set_facecolor("#111")
fig.patch.set_facecolor("#111")

# Coloured borders (drawn as filled rectangles behind each image)
BORDER = 3
half   = ZOOM * THUMB / 2
for i, (x, y) in enumerate(coords):
    col = cluster_color(lbl[i])
    ax.add_patch(plt.Rectangle(
        (x - half - BORDER, y - half - BORDER),
        2 * half + 2 * BORDER, 2 * half + 2 * BORDER,
        color=col, zorder=1, linewidth=0,
    ))

# Thumbnail images
for i, (thumb, (x, y)) in enumerate(zip(thumbs, coords)):
    ab = AnnotationBbox(OffsetImage(thumb, zoom=ZOOM), (x, y),
                        frameon=False, zorder=2)
    ax.add_artist(ab)

# Index + panel labels (drawn on top)
for i, (r, (x, y)) in enumerate(zip(records, coords)):
    ax.text(x, y - half - BORDER - 2,
            f"#{r['index']} {_short_panel(r['panel'])}",
            fontsize=3.8, color="white", ha="center", va="top",
            zorder=3, alpha=0.85)

ax.autoscale_view()
margin = THUMB * 1.2
ax.set_xlim(coords[:, 0].min() - margin, coords[:, 0].max() + margin)
ax.set_ylim(coords[:, 1].min() - margin, coords[:, 1].max() + margin)
ax.axis("off")

nc = _state["n_clusters"]
nn = _state["n_noise"]
ax.set_title(
    f"Motif similarity map  ·  {N} crops  ·  CLIP {CLIP_MODEL} ({PREPROCESS_MODE})  ·  "
    f"{nc} HDBSCAN clusters + {nn} unclustered",
    color="white", fontsize=11, pad=10,
)

# Legend
legend_handles = [mpatches.Patch(color=_NOISE_COL, label=f"unclustered ({nn})")]
for c in sorted(set(lbl)):
    if c == -1: continue
    cnt = int((lbl == c).sum())
    legend_handles.append(mpatches.Patch(color=cluster_color(c), label=f"C{c} (n={cnt})"))
ax.legend(handles=legend_handles, loc="lower right", fontsize=7,
          facecolor="#222", labelcolor="white", framealpha=0.85,
          ncols=max(1, len(legend_handles) // 14))

plt.tight_layout()

# Save high-res copy next to the notebook
out_png = CACHE_DIR / f"motif_scatter_{PREPROCESS_MODE}.png"
fig.savefig(out_png, dpi=DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"Saved → {out_png}")
plt.show()


In [ ]:
# ── Cell 7: cosine similarity heatmap ─────────────────────────────────────
# Re-run after adjusting clustering in cell 5.

lbl = labels()
order       = np.argsort(lbl)
sim_sorted  = sim_matrix[np.ix_(order, order)]
sorted_lbls = lbl[order]

# Cluster boundary positions
boundaries = [0]
for i in range(1, N):
    if sorted_lbls[i] != sorted_lbls[i - 1]:
        boundaries.append(i)
boundaries.append(N)

fig_size = min(18, max(8, N / 14))
fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))
im = ax.imshow(sim_sorted, cmap="inferno", vmin=0.5, vmax=1.0, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="cosine similarity")

for b in boundaries[1:-1]:
    ax.axhline(b - 0.5, color="cyan", lw=0.5, alpha=0.55)
    ax.axvline(b - 0.5, color="cyan", lw=0.5, alpha=0.55)

tick_pos  = [(boundaries[i] + boundaries[i + 1]) / 2 for i in range(len(boundaries) - 1)]
tick_lbls = ["noise" if sorted_lbls[int(p)] == -1 else f"C{sorted_lbls[int(p)]}"
             for p in tick_pos]
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lbls, rotation=60, fontsize=7)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_lbls, fontsize=7)
ax.set_title(f"Pairwise cosine similarity — sorted by cluster  [{PREPROCESS_MODE}]", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# ── Cell 8: cluster gallery ────────────────────────────────────────────────
# Re-run after adjusting clustering in cell 5.

lbl          = labels()
GALLERY_THUMB = 80
MAX_COLS      = 20
cluster_ids   = sorted(c for c in set(lbl) if c != -1) + ([-1] if -1 in lbl else [])

for c in cluster_ids:
    idx_in_c = [i for i, l in enumerate(lbl) if l == c]
    # Sort by similarity to centroid (most representative first)
    centroid = embeddings[idx_in_c].mean(axis=0)
    centroid /= np.linalg.norm(centroid) + 1e-8
    order_c  = np.argsort(-(embeddings[idx_in_c] @ centroid))
    idx_in_c = [idx_in_c[j] for j in order_c]

    tag   = f"Cluster {c}" if c != -1 else "Unclustered (noise)"
    color = cluster_color(c)
    n     = len(idx_in_c)
    n_cols = min(n, MAX_COLS)
    n_rows = (n + n_cols - 1) // n_cols
    fig_w  = n_cols * (GALLERY_THUMB / 72)
    fig_h  = n_rows * (GALLERY_THUMB / 72) + 0.5

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)
    fig.patch.set_facecolor("#1a1a1a")
    fig.suptitle(f"{tag}  (n={n})", color="white", fontsize=9, x=0.02, ha="left", y=1.01)

    for row in axes:
        for ax in row:
            ax.axis("off"); ax.set_facecolor("#1a1a1a")

    for j, gi in enumerate(idx_in_c):
        r_ax = axes[j // n_cols][j % n_cols]
        r_ax.imshow(Image.open(records[gi]["path"]).convert("RGB"))
        r_ax.set_title(
            f"#{records[gi]['index']} {_short_panel(records[gi]['panel'])}",
            fontsize=4.5, color="white", pad=1.5,
        )
        for spine in r_ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(1.5); spine.set_visible(True)

    plt.tight_layout(pad=0.2)
    plt.show()


In [ ]:
# ── Cell 9: nearest-neighbour explorer ────────────────────────────────────

TOP_K = 12

def _label(r, i):
    return f"{_short_panel(r['panel'])} / #{r['index']:03d} {r['scale']} (iou={r['pred_iou']:.2f})"

picker = widgets.Dropdown(
    options=[(_label(r, i), i) for i, r in enumerate(records)],
    value=0,
    description="Query:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "55px"},
)
w_k = widgets.IntSlider(
    min=4, max=40, step=2, value=TOP_K,
    description="Top K:",
    continuous_update=False,
    style={"description_width": "55px"},
    layout=widgets.Layout(width="40%"),
)
w_filter_cluster = widgets.Checkbox(
    value=False,
    description="Same cluster only",
    style={"description_width": "initial"},
)
out_nn = widgets.Output()

def _show_nn(query_idx, k, same_cluster_only):
    out_nn.clear_output(wait=True)
    lbl  = labels()
    sims = sim_matrix[query_idx].copy()
    sims[query_idx] = -1

    if same_cluster_only and lbl[query_idx] != -1:
        sims[lbl != lbl[query_idx]] = -1

    top_idx = np.argsort(-sims)[:k]
    top_sim = sims[top_idx]

    n_cols = min(k, 8)
    n_rows = 1 + (k + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.8),
                             squeeze=False)
    fig.patch.set_facecolor("#1a1a1a")
    for row in axes:
        for ax in row:
            ax.axis("off"); ax.set_facecolor("#1a1a1a")

    qr = records[query_idx]
    axes[0][0].imshow(Image.open(qr["path"]).convert("RGB"))
    qlbl = lbl[query_idx]
    axes[0][0].set_title(
        f"QUERY  #{qr['index']} {qr['scale']}\n{_short_panel(qr['panel'])}\n"
        f"{'C'+str(qlbl) if qlbl != -1 else 'noise'}",
        fontsize=6, color="white", pad=2,
    )
    for spine in axes[0][0].spines.values():
        spine.set_edgecolor("gold"); spine.set_linewidth(2); spine.set_visible(True)
    for col in range(1, n_cols):
        axes[0][col].set_visible(False)

    for j, (ni, sv) in enumerate(zip(top_idx, top_sim)):
        row = 1 + j // n_cols
        col = j % n_cols
        nr  = records[ni]
        nlbl = lbl[ni]
        axes[row][col].imshow(Image.open(nr["path"]).convert("RGB"))
        axes[row][col].set_title(
            f"sim={sv:.3f}\n#{nr['index']} {nr['scale']}\n{_short_panel(nr['panel'])}",
            fontsize=5, color="white", pad=2,
        )
        for spine in axes[row][col].spines.values():
            spine.set_edgecolor(cluster_color(nlbl)); spine.set_linewidth(1.5); spine.set_visible(True)

    plt.suptitle(
        f"Top-{k} nearest neighbours  [{PREPROCESS_MODE}]",
        color="white", fontsize=9, y=1.01,
    )
    plt.tight_layout(pad=0.4)
    with out_nn:
        plt.show()

def _on_change(_):
    _show_nn(picker.value, w_k.value, w_filter_cluster.value)

picker.observe(_on_change, names="value")
w_k.observe(_on_change, names="value")
w_filter_cluster.observe(_on_change, names="value")

display(
    widgets.HTML("<h3 style='margin-bottom:4px'>Nearest-neighbour explorer</h3>"),
    picker,
    widgets.HBox([w_k, w_filter_cluster]),
    out_nn,
)
_show_nn(picker.value, w_k.value, w_filter_cluster.value)
